In [1]:
# Load libraries used to aggregate the current season roster.
from pathlib import Path

import numpy as np
import pandas as pd
from itertools import combinations

In [2]:
# Define current-season input, output, and target season.
INPUT_FILE = Path("../data/raw/PlayerStatistics.csv")
OUTPUT_FILE = Path("../data/interim/nba_2025_26_player_season_totals_with_ending_team.csv")

processed_data_path = Path("../data/processed")

TARGET_SEASON = "2025-26"

In [3]:
# Load current player box scores.
df = pd.read_csv(INPUT_FILE, low_memory=False)

In [4]:
# Parse dates and assign each row to an NBA season.
df["gameDate"] = pd.to_datetime(df["gameDate"], errors="coerce")

season_start_year = (df["gameDate"].dt.year - (df["gameDate"].dt.month < 7).astype(int)).astype("Int64")

df["Season"] = season_start_year.astype("string") + "-" + (season_start_year + 1).astype("string").str[-2:]

In [5]:
# Keep games from the target season.
season_games = df.loc[df["Season"].eq(TARGET_SEASON)].copy()

print(f"{TARGET_SEASON} rows before season-type filtering: " f"{len(season_games):,}")

2025-26 rows before season-type filtering: 37,576


In [6]:
print("TARGET_SEASON:", repr(TARGET_SEASON))

print("\nDate parsing:")
print("Rows:", len(df))
print("Missing/invalid dates:", df["gameDate"].isna().sum())
print("Minimum date:", df["gameDate"].min())
print("Maximum date:", df["gameDate"].max())

print("\nLatest generated seasons:")
print(df["Season"].value_counts().sort_index().tail(10))

TARGET_SEASON: '2025-26'

Date parsing:
Rows: 1669922
Missing/invalid dates: 3513
Minimum date: 1946-11-26 19:00:00
Maximum date: 2026-06-13 20:30:00

Latest generated seasons:
Season
2016-17    37003
2017-18    35743
2018-19    35375
2019-20    26828
2020-21    38628
2021-22    36602
2022-23    36512
2023-24    37300
2024-25    37779
2025-26    37576
Name: count, dtype: Int64


In [7]:
# Keep regular-season games only.
regular_season = season_games.loc[season_games["gameType"] == "Regular Season"]

In [8]:
regular_season.head()

,firstName,lastName,personId,gameId,gameDateTimeEst,playerteamCity,playerteamName,opponentteamCity,opponentteamName,gameType,...,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,playerteamId,opponentteamId,comment,startingPosition,gameDate,Season
2578,Dwight,Powell,203939.0,22501193,2026-04-12 20:30:00,Dallas,Mavericks,Chicago,Bulls,Regular Season,...,12.0,4.0,0.0,15.0,1.610613e+09,1.610613e+09,NaN,C,2026-04-12 20:30:00,2025-26
2579,Lachlan,Olbrich,1642950.0,22501193,2026-04-12 20:30:00,Chicago,Bulls,Dallas,Mavericks,Regular Season,...,15.0,2.0,1.0,-11.0,1.610613e+09,1.610613e+09,NaN,C,2026-04-12 20:30:00,2025-26
2580,Ryan,Nembhard,1642948.0,22501193,2026-04-12 20:30:00,Dallas,Mavericks,Chicago,Bulls,Regular Season,...,9.0,2.0,4.0,23.0,1.610613e+09,1.610613e+09,NaN,G,2026-04-12 20:30:00,2025-26
2581,Mouhamadou,Gueye,1631338.0,22501193,2026-04-12 20:30:00,Chicago,Bulls,Dallas,Mavericks,Regular Season,...,3.0,2.0,2.0,-14.0,1.610613e+09,1.610613e+09,NaN,NaN,2026-04-12 20:30:00,2025-26
2582,Yuki,Kawamura,1642530.0,22501193,2026-04-12 20:30:00,Chicago,Bulls,Dallas,Mavericks,Regular Season,...,2.0,2.0,0.0,-2.0,1.610613e+09,1.610613e+09,NaN,NaN,2026-04-12 20:30:00,2025-26


In [9]:
regular_season.tail()

,firstName,lastName,personId,gameId,gameDateTimeEst,playerteamCity,playerteamName,opponentteamCity,opponentteamName,gameType,...,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,playerteamId,opponentteamId,comment,startingPosition,gameDate,Season
35033,Alex,Caruso,1627936.0,22500001,2025-10-21 19:30:00,Oklahoma City,Thunder,Houston,Rockets,Regular Season,...,2.0,2.0,0.0,-15.0,1.610613e+09,1.610613e+09,NaN,NaN,2025-10-21 19:30:00,2025-26
35034,Tari,Eason,1631106.0,22500001,2025-10-21 19:30:00,Houston,Rockets,Oklahoma City,Thunder,Regular Season,...,6.0,1.0,4.0,-4.0,1.610613e+09,1.610613e+09,NaN,NaN,2025-10-21 19:30:00,2025-26
35035,Steven,Adams,203500.0,22500001,2025-10-21 19:30:00,Houston,Rockets,Oklahoma City,Thunder,Regular Season,...,13.0,4.0,3.0,2.0,1.610613e+09,1.610613e+09,NaN,C,2025-10-21 19:30:00,2025-26
35036,Jabari,Smith Jr.,1631095.0,22500001,2025-10-21 19:30:00,Houston,Rockets,Oklahoma City,Thunder,Regular Season,...,5.0,4.0,0.0,2.0,1.610613e+09,1.610613e+09,NaN,F,2025-10-21 19:30:00,2025-26
35037,Kevin,Durant,201142.0,22500001,2025-10-21 19:30:00,Houston,Rockets,Oklahoma City,Thunder,Regular Season,...,9.0,6.0,4.0,0.0,1.610613e+09,1.610613e+09,NaN,G,2025-10-21 19:30:00,2025-26


In [10]:
# Normalize minutes and mark actual appearances.
regular_season = regular_season.copy()

regular_season["numMinutes"] = pd.to_numeric(regular_season["numMinutes"], errors="coerce")

appeared = regular_season.loc[regular_season["numMinutes"].notna()].copy()

print(f"Player appearances: {len(appeared):,}")

Player appearances: 26,730


In [11]:
# Define counting statistics to aggregate by player.
sum_columns = [
    "numMinutes",
    "points",
    "assists",
    "reboundsOffensive",
    "reboundsDefensive",
    "reboundsTotal",
    "fieldGoalsAttempted",
    "fieldGoalsMade",
    "threePointersAttempted",
    "threePointersMade",
    "freeThrowsAttempted",
    "freeThrowsMade",
    "steals",
    "blocks",
    "turnovers",
    "foulsPersonal",
    "plusMinusPoints",
]

sum_columns = [column for column in sum_columns if column in appeared.columns]

for column in sum_columns:
    appeared[column] = pd.to_numeric(appeared[column], errors="coerce")

In [12]:
# Create consistent player display names.
appeared["PLAYER_NAME"] = (appeared["firstName"].fillna("").str.strip() + " " + appeared["lastName"].fillna("").str.strip()).str.strip()

appeared["TEAM_FULL_NAME"] = (
    appeared["playerteamCity"].fillna("").str.strip() + " " + appeared["playerteamName"].fillna("").str.strip()
).str.strip()

In [13]:
# Aggregate current-season totals by player.
player_group = appeared.groupby("personId", as_index=False)

season_totals = player_group.agg(
    PLAYER_NAME=("PLAYER_NAME", "last"),
    GP=("gameId", "nunique"),
    FIRST_GAME=("gameDate", "min"),
    LAST_GAME=("gameDate", "max"),
    TEAM_STINTS=("TEAM_FULL_NAME", "nunique"),
)

summed_stats = appeared.groupby("personId")[sum_columns].sum(min_count=1).reset_index()

season_totals = season_totals.merge(summed_stats, on="personId", how="left", validate="one_to_one")

season_totals = season_totals.rename(
    columns={
        "personId": "PLAYER_ID",
        "numMinutes": "MIN",
        "points": "PTS",
        "assists": "AST",
        "reboundsOffensive": "OREB",
        "reboundsDefensive": "DREB",
        "reboundsTotal": "REB",
        "fieldGoalsAttempted": "FGA",
        "fieldGoalsMade": "FGM",
        "threePointersAttempted": "FG3A",
        "threePointersMade": "FG3M",
        "freeThrowsAttempted": "FTA",
        "freeThrowsMade": "FTM",
        "steals": "STL",
        "blocks": "BLK",
        "turnovers": "TOV",
        "foulsPersonal": "PF",
        "plusMinusPoints": "PLUS_MINUS",
    }
)

In [14]:
# Calculate shooting percentages safely when attempts are zero.
def safe_percentage(made: pd.Series, attempted: pd.Series) -> pd.Series:
    return np.where(attempted.gt(0), made / attempted, np.nan)


if {"FGM", "FGA"}.issubset(season_totals.columns):
    season_totals["FG_PCT"] = safe_percentage(season_totals["FGM"], season_totals["FGA"])

if {"FG3M", "FG3A"}.issubset(season_totals.columns):
    season_totals["FG3_PCT"] = safe_percentage(season_totals["FG3M"], season_totals["FG3A"])

if {"FTM", "FTA"}.issubset(season_totals.columns):
    season_totals["FT_PCT"] = safe_percentage(season_totals["FTM"], season_totals["FTA"])

In [15]:
# Assign each player to the team from their latest appearance.
ending_team = (
    appeared.sort_values(["personId", "gameDate", "gameId"])
    .groupby("personId", as_index=False)
    .tail(1)[["personId", "playerteamCity", "playerteamName", "TEAM_FULL_NAME", "gameDate"]]
    .rename(
        columns={
            "personId": "PLAYER_ID",
            "playerteamCity": "END_TEAM_CITY",
            "playerteamName": "END_TEAM_NAME",
            "TEAM_FULL_NAME": "END_TEAM_FULL_NAME",
            "gameDate": "FINAL_APPEARANCE_DATE",
        }
    )
)

season_totals = season_totals.merge(ending_team, on="PLAYER_ID", how="left", validate="one_to_one")

season_totals["SEASON"] = TARGET_SEASON

In [16]:
# Calculate per-game statistics.
for total_column, per_game_column in {
    "MIN": "MIN_PER_GAME",
    "PTS": "PTS_PER_GAME",
    "REB": "REB_PER_GAME",
    "AST": "AST_PER_GAME",
    "STL": "STL_PER_GAME",
    "BLK": "BLK_PER_GAME",
    "TOV": "TOV_PER_GAME",
    "PF": "PF_PER_GAME",
}.items():
    if total_column in season_totals.columns:
        season_totals[per_game_column] = season_totals[total_column] / season_totals["GP"]

In [17]:
# Calculate turnover percentage.
season_totals["TURNOVER_PERCENTAGE"] = np.where(
    (season_totals["FGA"] + 0.44 * season_totals["FTA"] + season_totals["TOV"]) > 0,
    100 * season_totals["TOV"] / (season_totals["FGA"] + 0.44 * season_totals["FTA"] + season_totals["TOV"]),
    np.nan,
)

In [18]:
# Calculate true shooting percentage.
season_totals["TRUE_SHOOTING_PERCENTAGE"] = np.where(
    2 * (season_totals["FGA"] + 0.44 * season_totals["FTA"]) > 0,
    season_totals["PTS"] / (2 * (season_totals["FGA"] + 0.44 * season_totals["FTA"])),
    np.nan,
)

In [19]:
# Calculate aggregate game score.
season_totals["GAME_SCORE_TOTAL"] = (
    season_totals["PTS"]
    + 0.4 * season_totals["FGM"]
    - 0.7 * season_totals["FGA"]
    - 0.4 * (season_totals["FTA"] - season_totals["FTM"])
    + 0.7 * season_totals["OREB"]
    + 0.3 * season_totals["DREB"]
    + season_totals["STL"]
    + 0.7 * season_totals["AST"]
    + 0.7 * season_totals["BLK"]
    - 0.4 * season_totals["PF"]
    - season_totals["TOV"]
)

season_totals["GAME_SCORE_PER_36"] = np.where(
    season_totals["MIN"] > 0, 36 * season_totals["GAME_SCORE_TOTAL"] / season_totals["MIN"], np.nan
)

In [20]:
regular_season.columns

Index(['firstName', 'lastName', 'personId', 'gameId', 'gameDateTimeEst',
       'playerteamCity', 'playerteamName', 'opponentteamCity',
       'opponentteamName', 'gameType', 'gameLabel', 'gameSubLabel',
       'seriesGameNumber', 'win', 'home', 'numMinutes', 'points', 'assists',
       'blocks', 'steals', 'fieldGoalsAttempted', 'fieldGoalsMade',
       'fieldGoalsPercentage', 'threePointersAttempted', 'threePointersMade',
       'threePointersPercentage', 'freeThrowsAttempted', 'freeThrowsMade',
       'freeThrowsPercentage', 'reboundsDefensive', 'reboundsOffensive',
       'reboundsTotal', 'foulsPersonal', 'turnovers', 'plusMinusPoints',
       'playerteamId', 'opponentteamId', 'comment', 'startingPosition',
       'gameDate', 'Season'],
      dtype='object')

In [21]:
# Enrich current box scores with possession and opponent context.
def build_enriched_box_scores(regular_season):

    box_scores = regular_season.copy()

    # ---------------------------------------------------------
    # Validate the fixed source schema
    # ---------------------------------------------------------

    required_columns = {
        "personId",
        "gameId",
        "gameDateTimeEst",
        "playerteamId",
        "playerteamCity",
        "playerteamName",
        "numMinutes",
        "points",
        "assists",
        "reboundsOffensive",
        "reboundsDefensive",
        "reboundsTotal",
        "fieldGoalsAttempted",
        "fieldGoalsMade",
        "threePointersAttempted",
        "threePointersMade",
        "freeThrowsAttempted",
        "freeThrowsMade",
        "steals",
        "blocks",
        "turnovers",
        "foulsPersonal",
        "plusMinusPoints",
    }

    missing_columns = sorted(required_columns - set(box_scores.columns))

    if missing_columns:
        raise KeyError("regular_season is missing required columns: " f"{missing_columns}")

    # ---------------------------------------------------------
    # Add the columns used by the player-stat calculations.
    # ---------------------------------------------------------

    box_scores["minutesPlayed"] = pd.to_numeric(box_scores["numMinutes"], errors="coerce")

    box_scores["game_Date"] = pd.to_datetime(box_scores["gameDateTimeEst"], errors="coerce").dt.normalize()

    # ---------------------------------------------------------
    # Numeric types
    # ---------------------------------------------------------

    numeric_columns = [
        "personId",
        "playerteamId",
        "minutesPlayed",
        "points",
        "assists",
        "reboundsOffensive",
        "reboundsDefensive",
        "reboundsTotal",
        "fieldGoalsAttempted",
        "fieldGoalsMade",
        "threePointersAttempted",
        "threePointersMade",
        "freeThrowsAttempted",
        "freeThrowsMade",
        "steals",
        "blocks",
        "turnovers",
        "foulsPersonal",
        "plusMinusPoints",
    ]

    for column in numeric_columns:
        box_scores[column] = pd.to_numeric(box_scores[column], errors="coerce")

    # A player appeared when the source contains a valid minute value.
    box_scores["appeared"] = box_scores["minutesPlayed"].notna()

    # ---------------------------------------------------------
    # Starter indicator
    # ---------------------------------------------------------

    if "position" in box_scores.columns:
        box_scores["started"] = box_scores["position"].fillna("").astype(str).str.strip().ne("") & box_scores["appeared"]
    else:
        box_scores["started"] = False

    # ---------------------------------------------------------
    # Stable team key
    # ---------------------------------------------------------

    box_scores["team_key"] = box_scores["playerteamId"].astype("Int64").astype("string")

    # ---------------------------------------------------------
    # Data-quality checks before aggregation
    # ---------------------------------------------------------

    duplicate_player_games = box_scores.duplicated(subset=["gameId", "personId"], keep=False)

    if duplicate_player_games.any():
        duplicate_sample = box_scores.loc[duplicate_player_games, ["gameId", "personId", "team_key"]].head(10)

        raise ValueError("Duplicate player-game rows were found.\n" f"{duplicate_sample}")

    teams_per_game = box_scores.groupby("gameId")["team_key"].nunique()

    invalid_games = teams_per_game.loc[teams_per_game.ne(2)]

    if not invalid_games.empty:
        raise ValueError("Some games do not contain exactly two teams. " f"Sample:\n{invalid_games.head(10)}")

    # ---------------------------------------------------------
    # Aggregate player rows to team-game totals
    # ---------------------------------------------------------

    appeared_rows = box_scores.loc[box_scores["appeared"]].copy()

    def sum_with_minimum(series: pd.Series):
        return series.sum(min_count=1)

    team_games = appeared_rows.groupby(["gameId", "team_key"], as_index=False).agg(
        team_minutes=("minutesPlayed", sum_with_minimum),
        team_pts=("points", sum_with_minimum),
        team_fga=("fieldGoalsAttempted", sum_with_minimum),
        team_fgm=("fieldGoalsMade", sum_with_minimum),
        team_fg3a=("threePointersAttempted", sum_with_minimum),
        team_fg3m=("threePointersMade", sum_with_minimum),
        team_fta=("freeThrowsAttempted", sum_with_minimum),
        team_ftm=("freeThrowsMade", sum_with_minimum),
        team_tov=("turnovers", sum_with_minimum),
        team_orb=("reboundsOffensive", sum_with_minimum),
        team_drb=("reboundsDefensive", sum_with_minimum),
        team_trb=("reboundsTotal", sum_with_minimum),
    )

    # ---------------------------------------------------------
    # Recover opponent totals from the other team in each game
    # ---------------------------------------------------------

    total_suffixes = ["minutes", "pts", "fga", "fgm", "fg3a", "fg3m", "fta", "ftm", "tov", "orb", "drb", "trb"]

    for suffix in total_suffixes:
        team_column = f"team_{suffix}"
        opponent_column = f"opp_{suffix}"

        game_total = team_games.groupby("gameId")[team_column].transform("sum")

        team_games[opponent_column] = game_total - team_games[team_column]

    # Actual game length: 48 minutes in regulation,
    # 53 in one overtime, etc.
    team_games["game_minutes"] = team_games["team_minutes"] / 5

    # ---------------------------------------------------------
    # Possession estimates
    # ---------------------------------------------------------

    team_orb_opportunities = team_games["team_orb"] + team_games["opp_drb"]

    opponent_orb_opportunities = team_games["opp_orb"] + team_games["team_drb"]

    team_orb_share = np.where(team_orb_opportunities > 0, team_games["team_orb"] / team_orb_opportunities, np.nan)

    opp_orb_share = np.where(opponent_orb_opportunities > 0, team_games["opp_orb"] / opponent_orb_opportunities, np.nan)

    team_games["team_possession_estimate"] = (
        team_games["team_fga"]
        + 0.4 * team_games["team_fta"]
        - 1.07 * team_orb_share * (team_games["team_fga"] - team_games["team_fgm"])
        + team_games["team_tov"]
    )

    team_games["opp_possession_estimate"] = (
        team_games["opp_fga"]
        + 0.4 * team_games["opp_fta"]
        - 1.07 * opp_orb_share * (team_games["opp_fga"] - team_games["opp_fgm"])
        + team_games["opp_tov"]
    )

    team_games["estimated_possessions"] = (team_games["team_possession_estimate"] + team_games["opp_possession_estimate"]) / 2

    # ---------------------------------------------------------
    # Merge team-game context onto every player-game row
    # ---------------------------------------------------------

    enriched = box_scores.merge(team_games, on=["gameId", "team_key"], how="left", validate="many_to_one")

    # ---------------------------------------------------------
    # Player possession estimate
    # ---------------------------------------------------------

    enriched["estimatedPlayerPossessions"] = (
        enriched["estimated_possessions"] * enriched["minutesPlayed"] / enriched["game_minutes"].replace(0, np.nan)
    )

    # ---------------------------------------------------------
    # Additional player-game features
    # ---------------------------------------------------------

    enriched["twoPointersMade"] = enriched["fieldGoalsMade"] - enriched["threePointersMade"]

    enriched["twoPointersAttempted"] = enriched["fieldGoalsAttempted"] - enriched["threePointersAttempted"]

    enriched["gameScore"] = (
        enriched["points"]
        + 0.4 * enriched["fieldGoalsMade"]
        - 0.7 * enriched["fieldGoalsAttempted"]
        - 0.4 * (enriched["freeThrowsAttempted"] - enriched["freeThrowsMade"])
        + 0.7 * enriched["reboundsOffensive"]
        + 0.3 * enriched["reboundsDefensive"]
        + enriched["steals"]
        + 0.7 * enriched["assists"]
        + 0.7 * enriched["blocks"]
        - 0.4 * enriched["foulsPersonal"]
        - enriched["turnovers"]
    )

    # ---------------------------------------------------------
    # Team result
    # ---------------------------------------------------------

    valid_scores = enriched["team_pts"].notna() & enriched["opp_pts"].notna()

    enriched["win"] = pd.Series(pd.NA, index=enriched.index, dtype="boolean")

    enriched.loc[valid_scores, "win"] = enriched.loc[valid_scores, "team_pts"] > enriched.loc[valid_scores, "opp_pts"]

    return enriched

In [22]:
enriched_Box_Scores_2025_26 = build_enriched_box_scores(regular_season)

print("Rows:", f"{len(enriched_Box_Scores_2025_26):,}")

print("Player appearances:", f"{enriched_Box_Scores_2025_26['appeared'].sum():,}")

display(
    enriched_Box_Scores_2025_26[
        [
            "gameId",
            "game_Date",
            "personId",
            "firstName",
            "lastName",
            "playerteamCity",
            "playerteamName",
            "minutesPlayed",
            "estimated_possessions",
            "estimatedPlayerPossessions",
            "gameScore",
            "team_pts",
            "opp_pts",
            "win",
        ]
    ].head()
)

Rows: 32,259
Player appearances: 26,730


,gameId,game_Date,personId,firstName,lastName,playerteamCity,playerteamName,minutesPlayed,estimated_possessions,estimatedPlayerPossessions,gameScore,team_pts,opp_pts,win
0,22501193,2026-04-12,203939.0,Dwight,Powell,Dallas,Mavericks,24.23,111.614333,56.720119,10.6,149.0,128.0,True
1,22501193,2026-04-12,1642950.0,Lachlan,Olbrich,Chicago,Bulls,30.36,111.614333,71.186319,21.2,128.0,149.0,False
2,22501193,2026-04-12,1642948.0,Ryan,Nembhard,Dallas,Mavericks,38.17,111.614333,89.352330,26.5,149.0,128.0,True
3,22501193,2026-04-12,1631338.0,Mouhamadou,Gueye,Chicago,Bulls,28.42,111.614333,66.637523,11.0,128.0,149.0,False
4,22501193,2026-04-12,1642530.0,Yuki,Kawamura,Chicago,Bulls,14.48,111.614333,33.951841,8.1,128.0,149.0,False


In [23]:
# Save current-season player totals.
season_totals.to_csv(OUTPUT_FILE, index=False)

print(f"Players: {len(season_totals):,}")
print(f"Ending teams represented: " f"{season_totals['END_TEAM_FULL_NAME'].nunique()}")
print(f"Saved: {OUTPUT_FILE}")

Players: 587
Ending teams represented: 30
Saved: ..\data\interim\nba_2025_26_player_season_totals_with_ending_team.csv


In [24]:
# Aggregate the additional rate-stat inputs by player.
enriched = enriched_Box_Scores_2025_26.copy()

appearance_rows = enriched.loc[enriched["appeared"].eq(True)].copy()


def sum_min_count(series):
    return series.sum(min_count=1)


season_additional_stats = (
    appearance_rows.groupby("personId", as_index=False)
    .agg(
        # Exposure
        CALCULATED_GP=("gameId", "nunique"),
        CALCULATED_MIN=("minutesPlayed", sum_min_count),
        # Additive model inputs
        ESTIMATED_PLAYER_POSSESSIONS=("estimatedPlayerPossessions", sum_min_count),
        GAME_SCORE_TOTAL=("gameScore", sum_min_count),
        # Player counting totals
        CALCULATED_PTS=("points", sum_min_count),
        CALCULATED_AST=("assists", sum_min_count),
        CALCULATED_REB=("reboundsTotal", sum_min_count),
        CALCULATED_FGA=("fieldGoalsAttempted", sum_min_count),
        CALCULATED_FTA=("freeThrowsAttempted", sum_min_count),
        CALCULATED_STL=("steals", sum_min_count),
        CALCULATED_BLK=("blocks", sum_min_count),
        CALCULATED_TOV=("turnovers", sum_min_count),
        CALCULATED_PLUS_MINUS=("plusMinusPoints", sum_min_count),
        # Team/opponent context from games played
        TEAM_MINUTES_CONTEXT=("team_minutes", sum_min_count),
        TEAM_REB_CONTEXT=("team_trb", sum_min_count),
        OPP_REB_CONTEXT=("opp_trb", sum_min_count),
        POSSESSIONS_CONTEXT=("estimated_possessions", sum_min_count),
        OPP_FGA_CONTEXT=("opp_fga", sum_min_count),
        OPP_FG3A_CONTEXT=("opp_fg3a", sum_min_count),
    )
    .rename(columns={"personId": "PLAYER_ID"})
)

season_additional_stats["SEASON"] = "2025-26"

In [25]:
season_additional_stats["PLAYER_ID"] = pd.to_numeric(season_additional_stats["PLAYER_ID"], errors="coerce").astype("Int64")

In [26]:
# Join the enriched metrics to player season totals.
season_totals_enriched = season_totals.merge(
    season_additional_stats, on=["PLAYER_ID", "SEASON"], how="left", validate="one_to_one", indicator=True, suffixes=("", "_CALCULATED")
)

In [27]:
print("Original rows:", len(season_totals))
print("Merged rows:", len(season_totals_enriched))

Original rows: 587
Merged rows: 587


In [28]:
# Provide safe division for rate calculations.
def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")

    denominator = pd.to_numeric(denominator, errors="coerce")

    result = numerator / denominator

    return result.where(denominator.notna() & denominator.gt(0))


season_totals_enriched["TEAM_GAME_MINUTES_CONTEXT"] = season_totals_enriched["TEAM_MINUTES_CONTEXT"] / 5

season_totals_enriched["ESTIMATED_PLAYER_POSSESSIONS_PER_GAME"] = safe_divide(
    season_totals_enriched["ESTIMATED_PLAYER_POSSESSIONS"], season_totals_enriched["CALCULATED_GP"]
)

season_totals_enriched["POINTS_PER_100"] = 100 * safe_divide(
    season_totals_enriched["CALCULATED_PTS"], season_totals_enriched["ESTIMATED_PLAYER_POSSESSIONS"]
)

season_totals_enriched["ASSISTS_PER_100"] = 100 * safe_divide(
    season_totals_enriched["CALCULATED_AST"], season_totals_enriched["ESTIMATED_PLAYER_POSSESSIONS"]
)

season_totals_enriched["PLUS_MINUS_PER_100"] = 100 * safe_divide(
    season_totals_enriched["CALCULATED_PLUS_MINUS"], season_totals_enriched["ESTIMATED_PLAYER_POSSESSIONS"]
)

total_rebound_denominator = season_totals_enriched["CALCULATED_MIN"] * (
    season_totals_enriched["TEAM_REB_CONTEXT"] + season_totals_enriched["OPP_REB_CONTEXT"]
)

season_totals_enriched["TOTAL_REBOUND_PERCENTAGE"] = 100 * safe_divide(
    season_totals_enriched["CALCULATED_REB"] * season_totals_enriched["TEAM_GAME_MINUTES_CONTEXT"], total_rebound_denominator
)

steal_denominator = season_totals_enriched["CALCULATED_MIN"] * season_totals_enriched["POSSESSIONS_CONTEXT"]

season_totals_enriched["STEAL_PERCENTAGE"] = 100 * safe_divide(
    season_totals_enriched["CALCULATED_STL"] * season_totals_enriched["TEAM_GAME_MINUTES_CONTEXT"], steal_denominator
)

season_totals_enriched["OPP_TWO_POINT_ATTEMPTS_CONTEXT"] = (
    season_totals_enriched["OPP_FGA_CONTEXT"] - season_totals_enriched["OPP_FG3A_CONTEXT"]
)

block_denominator = season_totals_enriched["CALCULATED_MIN"] * season_totals_enriched["OPP_TWO_POINT_ATTEMPTS_CONTEXT"]

season_totals_enriched["BLOCK_PERCENTAGE"] = 100 * safe_divide(
    season_totals_enriched["CALCULATED_BLK"] * season_totals_enriched["TEAM_GAME_MINUTES_CONTEXT"], block_denominator
)

In [29]:
# Load the frozen historical production reference.
production_reference = pd.read_parquet(Path("../data/reference/draft_weight_production_reference.parquet"))

In [30]:
print(production_reference.shape)
print(production_reference.columns.tolist())

display(production_reference.describe().T)

(904, 10)
['net_season_points_per_100', 'net_season_true_shooting_percentage', 'net_season_assists_per_100', 'net_season_turnover_percentage_weighted_mean', 'net_season_total_rebound_percentage_weighted_mean', 'net_season_steal_percentage_weighted_mean', 'net_season_block_percentage_weighted_mean', 'net_season_plus_minus_per_100', 'net_season_game_score_average', 'on_court_production_differential']


,count,mean,std,min,25%,50%,75%,max
net_season_points_per_100,904.0,-1.165931e-02,6.496937,-18.958230,-4.377821,0.160133,4.296048,18.958230
net_season_true_shooting_percentage,904.0,-2.019116e-04,0.063205,-0.308976,-0.038546,0.000372,0.037998,0.308976
net_season_assists_per_100,904.0,1.649082e-02,3.242556,-12.404595,-1.981665,-0.021686,1.889026,12.404595
net_season_turnover_percentage_weighted_mean,904.0,9.801340e-03,4.952775,-17.145509,-3.129192,-0.066344,3.106769,17.145509
net_season_total_rebound_percentage_weighted_mean,904.0,6.266564e-02,4.877649,-17.800542,-2.748586,0.030474,2.748586,17.800542
net_season_steal_percentage_weighted_mean,904.0,-2.478061e-03,0.801519,-3.286844,-0.496032,0.007969,0.479029,3.286844
net_season_block_percentage_weighted_mean,904.0,9.338526e-03,1.577909,-10.172904,-0.683531,0.000000,0.685996,10.172904
net_season_plus_minus_per_100,904.0,6.628469e-02,6.709777,-33.448120,-2.924725,0.000000,3.160212,33.448120
net_season_game_score_average,904.0,4.834228e-02,4.486401,-16.071488,-2.696964,0.000000,2.740123,19.825337
on_court_production_differential,904.0,8.965297e-18,0.379211,-1.442699,-0.220232,-0.002340,0.221066,1.566790


In [31]:
# Standardize current production against the historical reference.
historical_reference_parameters = production_reference.agg(["mean", "std"]).T.rename(
    columns={"mean": "REFERENCE_MEAN", "std": "REFERENCE_STD_DDOF_1"}
)

historical_reference_parameters["REFERENCE_STD_DDOF_0"] = production_reference.std(ddof=0)

display(historical_reference_parameters)

,REFERENCE_MEAN,REFERENCE_STD_DDOF_1,REFERENCE_STD_DDOF_0
net_season_points_per_100,-1.165931e-02,6.496937,6.493343
net_season_true_shooting_percentage,-2.019116e-04,0.063205,0.063170
net_season_assists_per_100,1.649082e-02,3.242556,3.240762
net_season_turnover_percentage_weighted_mean,9.801340e-03,4.952775,4.950035
net_season_total_rebound_percentage_weighted_mean,6.266564e-02,4.877649,4.874951
net_season_steal_percentage_weighted_mean,-2.478061e-03,0.801519,0.801075
net_season_block_percentage_weighted_mean,9.338526e-03,1.577909,1.577036
net_season_plus_minus_per_100,6.628469e-02,6.709777,6.706065
net_season_game_score_average,4.834228e-02,4.486401,4.483919
on_court_production_differential,8.965297e-18,0.379211,0.379001


In [32]:
# Save the enriched roster source used by the app build.
season_totals_enriched.to_csv("../data/interim/season_Totals_Enriched.csv")

In [33]:
season_totals_enriched.columns.tolist()

['PLAYER_ID',
 'PLAYER_NAME',
 'GP',
 'FIRST_GAME',
 'LAST_GAME',
 'TEAM_STINTS',
 'MIN',
 'PTS',
 'AST',
 'OREB',
 'DREB',
 'REB',
 'FGA',
 'FGM',
 'FG3A',
 'FG3M',
 'FTA',
 'FTM',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PLUS_MINUS',
 'FG_PCT',
 'FG3_PCT',
 'FT_PCT',
 'END_TEAM_CITY',
 'END_TEAM_NAME',
 'END_TEAM_FULL_NAME',
 'FINAL_APPEARANCE_DATE',
 'SEASON',
 'MIN_PER_GAME',
 'PTS_PER_GAME',
 'REB_PER_GAME',
 'AST_PER_GAME',
 'STL_PER_GAME',
 'BLK_PER_GAME',
 'TOV_PER_GAME',
 'PF_PER_GAME',
 'TURNOVER_PERCENTAGE',
 'TRUE_SHOOTING_PERCENTAGE',
 'GAME_SCORE_TOTAL',
 'GAME_SCORE_PER_36',
 'CALCULATED_GP',
 'CALCULATED_MIN',
 'ESTIMATED_PLAYER_POSSESSIONS',
 'GAME_SCORE_TOTAL_CALCULATED',
 'CALCULATED_PTS',
 'CALCULATED_AST',
 'CALCULATED_REB',
 'CALCULATED_FGA',
 'CALCULATED_FTA',
 'CALCULATED_STL',
 'CALCULATED_BLK',
 'CALCULATED_TOV',
 'CALCULATED_PLUS_MINUS',
 'TEAM_MINUTES_CONTEXT',
 'TEAM_REB_CONTEXT',
 'OPP_REB_CONTEXT',
 'POSSESSIONS_CONTEXT',
 'OPP_FGA_CONTEXT',
 'OPP_FG3A_CONTEXT',


In [34]:
# Create one row of additive components per player.
package_components = season_totals_enriched[
    [
        "PLAYER_ID",
        "PLAYER_NAME",
        "END_TEAM_FULL_NAME",
        "CALCULATED_GP",
        "CALCULATED_MIN",
        "CALCULATED_PTS",
        "CALCULATED_AST",
        "CALCULATED_FGA",
        "CALCULATED_FTA",
        "CALCULATED_PLUS_MINUS",
        "ESTIMATED_PLAYER_POSSESSIONS",
        "GAME_SCORE_TOTAL",
        "TURNOVER_PERCENTAGE",
        "TOTAL_REBOUND_PERCENTAGE",
        "STEAL_PERCENTAGE",
        "BLOCK_PERCENTAGE",
    ]
].copy()

package_components["MIN_X_TOV_PCT"] = package_components["CALCULATED_MIN"] * package_components["TURNOVER_PERCENTAGE"]

package_components["MIN_X_TRB_PCT"] = package_components["CALCULATED_MIN"] * package_components["TOTAL_REBOUND_PERCENTAGE"]

package_components["MIN_X_STL_PCT"] = package_components["CALCULATED_MIN"] * package_components["STEAL_PERCENTAGE"]

package_components["MIN_X_BLK_PCT"] = package_components["CALCULATED_MIN"] * package_components["BLOCK_PERCENTAGE"]

In [35]:
# Save reusable player package components.
component_columns = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "END_TEAM_FULL_NAME",
    "CALCULATED_GP",
    "CALCULATED_MIN",
    "CALCULATED_PTS",
    "CALCULATED_AST",
    "CALCULATED_FGA",
    "CALCULATED_FTA",
    "CALCULATED_PLUS_MINUS",
    "ESTIMATED_PLAYER_POSSESSIONS",
    "GAME_SCORE_TOTAL",
    "MIN_X_TOV_PCT",
    "MIN_X_TRB_PCT",
    "MIN_X_STL_PCT",
    "MIN_X_BLK_PCT",
]

package_components[component_columns].to_parquet(processed_data_path / "player_package_components.parquet", index=False)

In [36]:
# Enumerate one- and two-player packages by team.
ADDITIVE_COLUMNS = [
    "CALCULATED_GP",
    "CALCULATED_MIN",
    "CALCULATED_PTS",
    "CALCULATED_AST",
    "CALCULATED_FGA",
    "CALCULATED_FTA",
    "CALCULATED_PLUS_MINUS",
    "ESTIMATED_PLAYER_POSSESSIONS",
    "GAME_SCORE_TOTAL",
    "MIN_X_TOV_PCT",
    "MIN_X_TRB_PCT",
    "MIN_X_STL_PCT",
    "MIN_X_BLK_PCT",
]


def build_team_package_catalog(team_roster):
    records = []

    for package_size in (1, 2):
        for indices in combinations(team_roster.index, package_size):
            package = team_roster.loc[list(indices)]

            record = {
                "END_TEAM_FULL_NAME": package["END_TEAM_FULL_NAME"].iloc[0],
                "PACKAGE_SIZE": package_size,
                "PLAYER_IDS": tuple(package["PLAYER_ID"].astype(int)),
                "PLAYER_NAMES": tuple(package["PLAYER_NAME"].astype(str)),
            }

            totals = package[ADDITIVE_COLUMNS].sum()

            for column in ADDITIVE_COLUMNS:
                record[column] = totals[column]

            records.append(record)

    return pd.DataFrame(records)


package_catalog = (
    package_components.groupby("END_TEAM_FULL_NAME", group_keys=False).apply(build_team_package_catalog).reset_index(drop=True)
)

package_catalog.to_parquet(processed_data_path / "team_player_package_catalog.parquet", index=False)

C:\Users\cmows\AppData\Local\Temp\ipykernel_53688\1069395657.py:44: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  package_components.groupby("END_TEAM_FULL_NAME", group_keys=False).apply(build_team_package_catalog).reset_index(drop=True)


In [37]:
# Create a small package example for calculation checks.
test_player_names = ["Christian Koloko", "Baylor Scheierman"]

test_package = package_components.loc[package_components["PLAYER_NAME"].isin(test_player_names)].copy()

package_vector = test_package[ADDITIVE_COLUMNS].sum(min_count=1)

display(package_vector)

CALCULATED_GP                     104.000000
CALCULATED_MIN                   1765.620000
CALCULATED_PTS                    500.000000
CALCULATED_AST                    134.000000
CALCULATED_FGA                    407.000000
CALCULATED_FTA                     46.000000
CALCULATED_PLUS_MINUS             186.000000
ESTIMATED_PLAYER_POSSESSIONS     3540.029437
GAME_SCORE_TOTAL                  470.600000
MIN_X_TOV_PCT                   22391.635225
MIN_X_TRB_PCT                   18933.886565
MIN_X_STL_PCT                    2784.652597
MIN_X_BLK_PCT                    2654.218497
dtype: float64

In [38]:
# Manually calculate the example package rates.
points_per_100 = 100 * package_vector["CALCULATED_PTS"] / package_vector["ESTIMATED_PLAYER_POSSESSIONS"]

true_shooting = package_vector["CALCULATED_PTS"] / (2 * (package_vector["CALCULATED_FGA"] + 0.44 * package_vector["CALCULATED_FTA"]))

assists_per_100 = 100 * package_vector["CALCULATED_AST"] / package_vector["ESTIMATED_PLAYER_POSSESSIONS"]

turnover_percentage = package_vector["MIN_X_TOV_PCT"] / package_vector["CALCULATED_MIN"]

total_rebound_percentage = package_vector["MIN_X_TRB_PCT"] / package_vector["CALCULATED_MIN"]

steal_percentage = package_vector["MIN_X_STL_PCT"] / package_vector["CALCULATED_MIN"]

block_percentage = package_vector["MIN_X_BLK_PCT"] / package_vector["CALCULATED_MIN"]

plus_minus_per_100 = 100 * package_vector["CALCULATED_PLUS_MINUS"] / package_vector["ESTIMATED_PLAYER_POSSESSIONS"]

game_score_average = package_vector["GAME_SCORE_TOTAL"] / package_vector["CALCULATED_GP"]

In [39]:
# Convert an additive package vector into model metrics.
def safe_scalar_divide(numerator, denominator):
    if pd.isna(numerator) or pd.isna(denominator) or denominator <= 0:
        return np.nan

    return float(numerator) / float(denominator)


def package_vector_to_metrics(package_vector):
    possessions = package_vector["ESTIMATED_PLAYER_POSSESSIONS"]

    minutes = package_vector["CALCULATED_MIN"]

    true_shooting_denominator = 2 * (package_vector["CALCULATED_FGA"] + 0.44 * package_vector["CALCULATED_FTA"])

    return {
        "season_points_per_100": (100 * safe_scalar_divide(package_vector["CALCULATED_PTS"], possessions)),
        "season_true_shooting_percentage": (safe_scalar_divide(package_vector["CALCULATED_PTS"], true_shooting_denominator)),
        "season_assists_per_100": (100 * safe_scalar_divide(package_vector["CALCULATED_AST"], possessions)),
        "season_turnover_percentage_weighted_mean": (safe_scalar_divide(package_vector["MIN_X_TOV_PCT"], minutes)),
        "season_total_rebound_percentage_weighted_mean": (safe_scalar_divide(package_vector["MIN_X_TRB_PCT"], minutes)),
        "season_steal_percentage_weighted_mean": (safe_scalar_divide(package_vector["MIN_X_STL_PCT"], minutes)),
        "season_block_percentage_weighted_mean": (safe_scalar_divide(package_vector["MIN_X_BLK_PCT"], minutes)),
        "season_plus_minus_per_100": (100 * safe_scalar_divide(package_vector["CALCULATED_PLUS_MINUS"], possessions)),
        "season_game_score_average": (safe_scalar_divide(package_vector["GAME_SCORE_TOTAL"], package_vector["CALCULATED_GP"])),
    }

In [40]:
test_package_metrics = package_vector_to_metrics(package_vector)

display(pd.Series(test_package_metrics).to_frame("value"))

,value
season_points_per_100,14.124176
season_true_shooting_percentage,0.585151
season_assists_per_100,3.785279
season_turnover_percentage_weighted_mean,12.682024
season_total_rebound_percentage_weighted_mean,10.723648
season_steal_percentage_weighted_mean,1.577153
season_block_percentage_weighted_mean,1.503278
season_plus_minus_per_100,5.254194
season_game_score_average,4.525000


In [41]:
# Save the final player component table.
component_output_columns = ["PLAYER_ID", "PLAYER_NAME", "END_TEAM_FULL_NAME", *ADDITIVE_COLUMNS]

package_components[component_output_columns].to_parquet(processed_data_path / "player_package_components.parquet", index=False)